# 00: Setup, orientation, and where everything lives

**This is the development-stack workshop (LATW `dev` branch).** The
tutorials here target the *development* versions of the LISA Analysis
Tools packages — the ones installed by `LISAanalysistools/install.sh`.
They are not the pip-released packages.

To set up the environment, clone LISAanalysistools and run its
installer (it clones every sibling repo — including this one — side by
side, checks out the development branches, and editable-installs
everything):

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

If instead you want the workshop on the **pip-released** packages
(`pip install lisaanalysistools eryn gbgpu bbhx fastemriwaveforms ...`),
use the [`main` branch](https://github.com/lisa-analysis-tools/LATW/tree/main).
That split is the repo's branch policy: `main` ↔ pip releases, `dev` ↔
the `install.sh` development stack.

### Prerequisite: LAPACK / LAPACKE

The compiled packages in the stack (GPUBackendTools, lisatools, BBHx, GBGPU,
FEW) **link LAPACKE at build time**, so it has to be present *before* you run
`install.sh`. GPUBackendTools owns the detection — the `GBT_LAPACKE_*` options
it exposes are shared by every compiled package in the chain — and by default
it looks for LAPACKE through `pkg-config`.

- **conda (all platforms, recommended):**
  ```bash
  conda create -n lisatools_env -y -c conda-forge --override-channels \
      cxx-compiler pkgconfig conda-forge/label/lapack_rc::liblapacke
  ```
- **macOS + Homebrew:** `brew install lapack`, then point `pkg-config` at the
  keg before building (there is a commented `PKG_CONFIG_PATH` block at the top
  of `install.sh` for exactly this):
  ```bash
  export PKG_CONFIG_PATH="$(brew --prefix lapack)/lib/pkgconfig:$PKG_CONFIG_PATH"
  ```
- **No system LAPACKE?** Let the build fetch + vendor a copy with
  `GBT_LAPACKE_FETCH=ON` (or
  `--config-settings=cmake.define.GBT_LAPACKE_FETCH=ON`).

This is a **build** dependency — distinct from the `OMP_NUM_THREADS` /
`OPENBLAS_NUM_THREADS` **runtime** thread-pinning in the setup cell below (that
caps how many threads BLAS uses at run time; this decides whether the C/CUDA
extensions compile at all). Full knob reference: `install.sh` and
`LISAanalysistools/DEVELOPMENT.md`; [`08`](08_BackendsAndDevWorkflow.ipynb)
covers how GBT's single detector feeds the whole stack.

In [1]:
import os

# Threading is pinned to 1 everywhere in this workshop (MPI-only policy;
# OMP-threaded kernels have caused out-of-memory kills on laptops).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

# A quick check that the development stack imports.
import lisatools, eryn, gbgpu, bbhx, gpubackendtools
for _pkg in (lisatools, eryn, gbgpu, bbhx, gpubackendtools):
    print(f"{_pkg.__name__:16s} {getattr(_pkg, '__version__', '?')}")
try:
    import few
    print(f"{'few':16s} {getattr(few, '__version__', '?')}")
except Exception as e:  # FEW is optional (SKIP_FEW=1)
    print(f"few not importable: {e!r}")

lisatools        1.2.8.post1.dev750+g636b5677d.d20260706
eryn             1.2.6
gbgpu            1.2.4.post1.dev113+gdc561ce8a.d20260711
bbhx             1.2.3.post1.dev43+gbd8674f2c.d20260619
gpubackendtools  0.1.1


few              2.0.0.post1.dev92+gf5f514166


## How to read this workshop

There are two tracks.

- **Informational notebooks** (`00`–`08`, this directory) are
  instructive: you read and run them. Every section opens with a
  **TL;DR** and one minimal cell; if you only want the surface, read the
  section headers and TL;DRs. Deeper material lives under
  *“Going deeper”* subsections for when you are building your own
  pipeline.
- **Exercise notebooks** (`further/`) are the workshop proper: **Tasks**
  (you write the code toward a stated goal) and **Questions** (discussion
  prompts about the physics/analysis). Worked solutions live in
  `further/answers/`.

The baseline the exercise track aims for: *if you want to do a real
research project on LISA data analysis, you should generally understand
and be able to work through these tutorials.*

## The ecosystem at a glance

The stack is several packages layered on a shared backend foundation.
One table, then a short tour.

| Package (import) | dev branch | pip name | Owns | Docs |
|---|---|---|---|---|
| **lisatools** (`lisatools`) | `dev` | `lisaanalysistools` | the central LISA-physics library: sensitivity, orbits, domains, the analysis container, the LISA response, and the global fit | [docs](https://mikekatz04.github.io/LISAanalysistools) |
| **eryn** (`eryn`) | `dev` | `eryn` | the MCMC sampler (fixed-dim, tempered, reversible-jump) everything sampling-related is built on | [docs](https://mikekatz04.github.io/Eryn) |
| **gbgpu** (`gbgpu`) | `dev` | `gbgpu` | Galactic-binary waveforms + fast GB likelihoods | [docs](https://mikekatz04.github.io/GBGPU) |
| **bbhx** (`bbhx`) | `dev` | `bbhx` | massive-black-hole-binary and stellar-origin-BHB waveforms (+ TDI-on-the-fly) | [docs](https://mikekatz04.github.io/BBHx) |
| **few** (`few`) | `gpu_backend` | `fastemriwaveforms` | fast EMRI waveforms | [docs](https://bhptoolkit.org/FastEMRIWaveforms) |
| **gpubackendtools** (`gpubackendtools`) | `spline` | `gpubackendtools` | the backend foundation (CPU/CUDA/JAX registry) + cubic-spline C++ | — |
| **fastlisaresponse** | — | `fastlisaresponse` | **deprecated** — the LISA response now lives in `lisatools.response` | — |

The dependency arrow points *down*: everything builds on
`gpubackendtools`; `gbgpu`/`bbhx` build on `lisatools`; `lisatools`
reaches *sideways* into `gbgpu`/`bbhx`/`few` (lazily) for its stock
waveforms. Cross-repo detail is in
`LISAanalysistools/docs/architecture-map.md`.

## A tour of `lisatools` (dependency order)

This is the spine the informational notebooks follow. Each layer builds
on the ones above it — the *small → large* story: the same objects that
compute a single SNR scale up to holding every residual of the global
fit.

- **`lisatools.domains`** — basis-tagged signal arrays: `TDSignal`,
  `FDSignal`, `STFTSignal`, `WDMSignal` (+ their `*Settings`). A signal
  always knows what basis it is in and can transform between them
  (`.fft()`, `.stft()`, `.wdmtransform()`). *(Notebook 04.)*
- **`lisatools.sensitivity`** — LISA noise: `get_sensitivity`,
  `SensitivityMatrix`, and stock curves/matrices
  (`A1TDISens`, `XYZ2SensitivityMatrix`, …). *(04.)*
- **`lisatools.detector`** — orbits and the constellation noise model:
  `Orbits`/`EqualArmlengthOrbits`, `LISAModel`. *(04.)*
- **`lisatools.analysiscontainer`** — `AnalysisContainer` (data + noise +
  a signal generator: `snr`, `inner_product`, `likelihood`) and
  `AnalysisContainerArray` (many of them, batched across the global
  fit's residuals). *(04.)*
- **`lisatools.response`** — the LISA TDI response: `ResponseWrapper`,
  `pyResponseTDI`, and the on-the-fly TDI family. *(05.)*
- **`lisatools.sources`** — per-source stock waveforms: GB, MBHB, EMRI,
  SOBHB. *(06.)*
- **`lisatools.diagnostic`** — the primitives underneath the container:
  `inner_product`, `snr`, `info_matrix`, `covariance`. *(04.)*
- **`lisatools.sampling`** — the glue to sampling: priors, stopping criteria, custom moves. *(07.)*
- **`lisatools.globalfit`** — the global fit: the stock configurations
  (`lisatools.globalfit.stock.erebor`), the recipe/stage/move machinery,
  the moves, the data processors, and the HDF5 output. *(01, 02, 03.)*
- **backends** — `lisatools.get_backend`, the `cutils` C++/CUDA layer,
  and the pure-JAX mirror in `lisatools.jax`. *(08.)*

## How to find things

When a tutorial isn't enough, these are the maps:

- **Developer guides** in `LISAanalysistools/docs/` — `conventions.md`
  (the coding rules that apply to every repo), `architecture-map.md`
  (where each capability lives across repos), `codebase-map.md`
  (lisatools' internal layout), `global-fit-launch.md` (running the
  global fit on ranks/GPUs), `stock-stages-and-moves.md` (the recipe
  layer). Each other repo has its own `docs/codebase-map.md`.
- **The Sphinx API docs** (linked in the table above) — the
  authoritative signatures and docstrings. The exercise tutorials link
  the exact pages you need under each Task's *“Useful documentation”*.
- **`grep`** — the code is the ground truth. Stock configurations live
  under `src/lisatools/globalfit/stock/erebor/`; waveforms under
  `src/lisatools/sources/`.

These LATW tutorials are also imported into the lisatools Sphinx docs,
so the workshop and the reference docs are the same material seen two
ways.

## Things *not* to use (and what to use instead)

A few symbols are still importable but are deprecated shims or dead
experiments. The tutorials never use them; neither should your code.

- `lisatools.datacontainer.DataResidualArray` — **deprecated** thin
  wrapper. Always use the `lisatools.domains` signal classes directly
  (`TDSignal` / `FDSignal` / `WDMSignal` / `STFTSignal`).
- `lisatools.mojito_detector` — retired; the live code is in
  `lisatools.detector`.
- `fastlisaresponse` (the `lisa-on-gpu` package) — deprecated; the
  response is in `lisatools.response`.

## Where to go next

1. **`01`** — run a small global fit in four lines and read every
   product out of it (the fastest way to see the whole pipeline).
2. **`02`** — the stock global fits **in depth**: the machinery under that
   one-liner — the data layer, the settings blocks, the recipe of moves,
   and how to add your own source class to the stack.
3. **`03`** — the stock global-fit **gallery**: a catalogue-style demo of
   every stock variant, once you know what the knobs mean.
4. **`04`–`08`** — the parts underneath, one at a time: the foundations
   (domains, sensitivity, the detector, the `AnalysisContainer`), the
   response, the source waveforms, sampling with Eryn, and the
   backends/dev workflow.
5. **`further/`** — the exercises, once you want to build your own
   analyses.